# Módulo 13 · Notebooks de prácticas

**Bootcamp IA 5 · KeepCoding · Marc Mayol Orell**

Notebook único con los 6 ejercicios prácticos del módulo. Cada sección puede ejecutarse de forma independiente, pero el bloque **Setup** debe lanzarse primero.

## Contenido

1. [Setup](#0)
2. [Notebook 01 · Tokens](#1)
3. [Notebook 02 · Parámetros](#2)
4. [Notebook 03 · Few-shot y CoT](#3)
5. [Notebook 04 · Sesgos y verificación](#4)
6. [Notebook 05 · Inyección y defensa](#5)
7. [Notebook 06 · Reto final](#6)

In [1]:
# Si estás en Colab o entorno limpio, descomenta:
!pip install openai tiktoken python-dotenv

### 2. Configurar la API key

Crea un archivo `.env` en la misma carpeta que este notebook con el siguiente contenido:

```
OPENAI_API_KEY=sk-tu-clave-aqui
```

**No subas nunca este archivo a GitHub.** Añade `.env` a tu `.gitignore`.

<a id='1'></a>
# Notebook 01 · Tokens

Vamos a ver físicamente cómo el modelo «trocea» un texto antes de procesarlo. Esto te ayudará a entender:

* Por qué el español es más caro que el inglés.
* Por qué frases similares pueden costar muy distinto.
* Cómo estimar el coste de un prompt antes de enviarlo.

In [11]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

def tokenize(texto):
  """Devuelve la lista de tokens y la decodificación de cada uno."""
  ids = enc.encode(texto)
  return [(t, enc.decode([t])) for t in ids]

# ejemplo

for token_id, token_text in tokenize("El prompt engineering es clave en el 2026."):
    print(f"{token_id:>7} -> {token_text!r}")

   4422 -> 'El'
  15226 -> ' prompt'
 142174 -> 'engineering'
    878 -> ' es'
  52470 -> ' clave'
    469 -> ' en'
    650 -> ' el'
    220 -> ' '
   1323 -> '202'
     21 -> '6'
     13 -> '.'


1.1 · Comparativa idiomas
Tokeniza la misma idea en distintos idiomas. ¿Cuántos tokens consume cada uno?

In [12]:
frases = [
    ("Inglés",   "Generative AI is changing the way we work."),
    ("Español",  "La IA generativa está cambiando la forma en que trabajamos."),
    ("Catalán",  "La IA generativa està canviant la forma en què treballem."),
    ("Francés",  "L'IA générative change notre façon de travailler."),
    ("Alemán",   "Generative KI verändert die Art, wie wir arbeiten."),
    ("Japonés",  "生成AIは私たちの働き方を変えつつある。"),
]
for idioma, frase in frases:
    n = len(enc.encode(frase))
    print(f"{idioma:<10} {n:>3} tokens · {len(frase):>3} caracteres · {frase}")

Inglés      10 tokens ·  42 caracteres · Generative AI is changing the way we work.
Español     14 tokens ·  59 caracteres · La IA generativa está cambiando la forma en que trabajamos.
Catalán     15 tokens ·  57 caracteres · La IA generativa està canviant la forma en què treballem.
Francés     11 tokens ·  49 caracteres · L'IA générative change notre façon de travailler.
Alemán      11 tokens ·  50 caracteres · Generative KI verändert die Art, wie wir arbeiten.
Japonés     17 tokens ·  20 caracteres · 生成AIは私たちの働き方を変えつつある。


## 1.2 · Coste estimado de un prompt

Estima cuánto te costaría procesar un texto largo con `gpt-4o-mini`. Precios aproximados (mayo 2026):

* Input: $0,15 por millón de tokens.

* Output: $0,60 por millón de tokens.

In [13]:
# Lee un texto de ejemplo (puedes pegar el tuyo)
texto = """La inteligencia artificial generativa está transformando profundamente
los procesos de trabajo en multitud de sectores. Desde la atención al cliente
hasta la investigación científica, los grandes modelos de lenguaje están
demostrando una capacidad sin precedentes para automatizar tareas que hasta
hace poco requerían intervención humana especializada."""

n_input = len(enc.encode(texto))
n_output_estimado = 300  # palabras esperadas en respuesta

precio_input  = 0.15 / 1_000_000  # $ por token input
precio_output = 0.60 / 1_000_000

coste_usd = n_input * precio_input + n_output_estimado * precio_output
print(f"Tokens input: {n_input}")
print(f"Tokens output estimados: {n_output_estimado}")
print(f"Coste estimado: ${coste_usd:.6f} (~{coste_usd*0.92:.6f} EUR)")

Tokens input: 61
Tokens output estimados: 300
Coste estimado: $0.000189 (~0.000174 EUR)


1.3 · Reto
Tokeniza un fragmento de tu propio código Python y otro fragmento de texto natural en español. ¿Cuál pesa más por carácter? Comenta el resultado.

In [15]:
codigo = """def calcular_iva(base, tipo=0.21):
    return base * (1 + tipo)"""

prosa = "El IVA general en España es del 21 por ciento."

In [16]:
print(f"Código: {len(enc.encode(codigo)):>3} tokens, {len(codigo):>3} caracteres "
      f"({len(enc.encode(codigo))/len(codigo):.3f} tokens/char)")
print(f"Prosa:  {len(enc.encode(prosa)):>3} tokens, {len(prosa):>3} caracteres "
      f"({len(enc.encode(prosa))/len(prosa):.3f} tokens/char)")

Código:  21 tokens,  63 caracteres (0.333 tokens/char)
Prosa:   12 tokens,  46 caracteres (0.261 tokens/char)
